In [ ]:
from example_models import get_linear_chain_1v, get_linear_chain_2v
from mxlpy import meta

# Metaprogramming

`mxlpy` can inspect the Python source of your model functions and transform them into other representations — whether that is runnable code in another language, a symbolic LaTeX description, or a diff between two model variants.

This works for any model whose rate and derived functions are **pure Python** (no `numpy`, no closures over mutable state). The source is parsed with Python's `ast` module and then re-emitted in the target language.

Use cases include:
- Sharing a model with collaborators who work in Rust, Julia, or TypeScript
- Generating a LaTeX methods section directly from code
- Tracking what changed between two model versions

## Code generation

`generate_mxlpy_code` round-trips your model back into `mxlpy` builder syntax. This is useful for serialising a model that was constructed programmatically (e.g. after fitting or after reading from SBML) into a standalone Python file that can be shared or version-controlled without any runtime dependencies on the original construction logic.

In [ ]:
print(meta.generate_mxlpy_code(get_linear_chain_1v()))

The `generate_model_code_*` family translates your model into a self-contained ODE function in the target language. Each function unpacks the variable vector, evaluates all derived quantities and rates inline, and returns the derivative vector — ready to hand to a native solver.

Currently supported targets: `Python`, `Rust`, `TypeScript`, `Julia`. The returned object exposes a `.model` attribute with the function source, plus `.derived` and `.inits` for the supporting code (see below).

In [ ]:
print(meta.generate_model_code_py(get_linear_chain_2v()).model)

In [ ]:
print(meta.generate_model_code_rs(get_linear_chain_2v()).model)

In [ ]:
print(meta.generate_model_code_ts(get_linear_chain_2v()).model)

In [ ]:
print(meta.generate_model_code_jl(get_linear_chain_2v()).model)

### Free parameters

By default, all parameters are baked into the generated function as constants. If you want to leave some parameters as inputs — for example to drive a parameter scan or a fitting loop in the target environment — pass their names as `free_parameters`. They will appear as additional function arguments instead of inlined literals.

## Derived quantities and initial conditions

Beyond the ODE function itself, a complete simulation needs two more pieces: a function that computes derived quantities from the current state (observables, intermediate fluxes), and the initial values for all variables.

The codegen object exposes these as `.derived` and `.inits` respectively. Both are emitted in the same target language as `.model`, so they can be dropped directly into the same file.

In [ ]:
codegen = meta.generate_model_code_py(get_linear_chain_2v(), free_parameters=["k_in"])

print(codegen.derived, end="\n\n")
print(codegen.inits)

## Pruning derived quantities

By default, the generated `.derived` function computes every derived quantity in the model. If you only need a subset — for example, you want to export a stripped-down function that outputs a single flux for a downstream optimiser — pass `derived_to_calculate` with the names you actually need. Quantities not in the list are omitted from the output function, and any intermediate computations required only by those omitted quantities are pruned automatically.

In [ ]:
codegen = meta.generate_model_code_py(
    get_linear_chain_2v(), derived_to_calculate=["v2"]
)

print(codegen.derived, end="\n\n")

## LaTeX export

`generate_model_code_latex` produces a structured LaTeX representation of your model — parameters, variables, rate equations, and stoichiometry — formatted as tables ready to paste into a manuscript or supplementary material.

Two optional arguments let you customise the output:
- `gls` — a `dict` mapping Python names to LaTeX names (e.g. `{"k_in": r"\k_{in}"`). Names without an entry are used verbatim.
- `long_name_cutoff` — names longer than this threshold are automatically abbreviated to avoid overflowing table columns.

Call `.as_tables()` on the result to get the individual LaTeX table strings.

In [ ]:
print(*meta.generate_model_code_latex(get_linear_chain_1v()).as_tables(), sep="\n\n")

## Exporting diffs

When you modify a model — changing a parameter value, swapping a rate law, adding a species — `generate_latex_diff` highlights exactly what changed between two model versions as a LaTeX table. Unchanged rows are shown in normal weight; added or modified rows are highlighted.

Pass `only_changes=True` to suppress unchanged rows entirely (useful when models are large and only a handful of values differ). With `only_changes=False` (shown below) every row is included, giving the full context alongside the highlighted changes.

In [ ]:
print(
    *meta.generate_latex_diff(
        get_linear_chain_1v(),
        get_linear_chain_1v().update_parameter("k_in", 2.0),
        only_changes=False,
    ).as_default(),
    sep="\n\n",
)

In [ ]:
print(
    *meta.generate_latex_diff(
        get_linear_chain_1v(),
        get_linear_chain_1v().update_parameter("k_in", 2.0),
        only_changes=False,
    ).as_default(),
    sep="\n\n",
)

## Exporting a document

`generate_latex_document` wraps any codegen result — either a single model or a diff — in a complete, compilable LaTeX document with the necessary preamble and package imports. This lets you go from a `Model` object to a `.tex` file you can compile directly with `pdflatex`, with no manual formatting needed.

In [ ]:
print(
    meta.generate_latex_document(
        meta.generate_model_code_latex(
            get_linear_chain_1v(),
        )
    )
)

In [ ]:
print(
    meta.generate_latex_document(
        meta.generate_latex_diff(
            get_linear_chain_1v(),
            get_linear_chain_1v().update_parameter("k_in", 2.0),
        )
    )
)

<div style="color: #ffffff; background-color: #04AA6D; padding: 3rem 1rem 3rem 1rem; box-sizing: border-box">
    <h2>First finish line</h2>
    With that you now know most of what you will need from a day-to-day basis about meta programming in mxlpy.
    <br />
    <br />
    Congratulations!
</div>

## Placeholders / error handling

If a model function uses constructs that `mxlpy` cannot translate - such as `numpy` calls, comprehensions, or external library calls - the LaTeX exporter inserts a red warning placeholder in place of that equation. The rest of the export is unaffected, so you still get a usable document for the parseable parts of your model.

In [ ]:
import numpy as np

from mxlpy import Model


def broken_fn() -> float:
    return np.sum(np.linalg.inv(np.array([[1.0, 2.0], [3.0, 4.0]])))  # type: ignore


print(
    meta.generate_model_code_latex(
        Model().add_derived("d1", broken_fn, args=[])
    ).as_default()
)

Code generation targets (Python, Rust, etc.) are stricter: they raise a `ValueError` rather than inserting a placeholder. Silent bad output is worse than a loud error when the generated function is part of a larger pipeline — a placeholder that compiles but computes nonsense would be very hard to debug downstream.

In [ ]:
try:
    meta.generate_model_code_py(Model().add_derived("d1", broken_fn, args=[]))
except ValueError as e:
    print("Errored:", e)

In [ ]:
try:
    meta.generate_model_code_rs(Model().add_derived("d1", broken_fn, args=[]))
except ValueError as e:
    print("Errored:", e)